In [10]:
import pandas as pd
import json
import re

def remove_version(key):
        """
        Remove the version suffix (e.g., "v1") from the key.
        This regex matches 'v' followed by one or more digits at the end of the string.
        """
        return re.sub(r'v\d+$', '', key)
# Define a function to extract ground truth
def extract_ground_truth(mapping_csv_path, blacklist_path, scopus_csv_path, default_output_path, institution_col="Primary Org Name"):
    """
    Extract ground truth from the CSV and merge with a mapping file.

    Filtering:
      - Exclude blacklisted institutions.
      - Exclude institutions with both "university" and "system".
      - Exclude institutions containing "Government of India".

    Groups non-null 'ROR ID's by paper ID.

    Parameters:
        mapping_csv_path (str): CSV with ROR mappings.
        blacklist_path (str): Text file with blacklisted organizations.
        scopus_csv_path (str): CSV with Scopus paper information.
        default_output_path (str): Directory path where the ground truth JSON should be saved.
        institution_col (str): Institution name column (default "Primary Org Name").

    Returns:
        dict: Mapping of paper ID to list of ROR IDs.
    """
    
    # Step 1: Load Scopus data
    print(scopus_csv_path)
    df = pd.read_csv(scopus_csv_path)
    
    # Step 2: Load blacklist
    with open(blacklist_path, "r", encoding="utf-8") as f:
        blacklist = [line.strip() for line in f if line.strip()]
    blacklist_lower = [org.lower() for org in blacklist]
    
    # Step 3: Create filters for unwanted records
    mask_blacklist = df[institution_col].str.lower().isin(blacklist_lower)
    mask_university_system = (
        df[institution_col].str.lower().str.contains('university', na=False) &
        df[institution_col].str.lower().str.contains('system', na=False)
    )
    mask_govt_india = df[institution_col].str.lower().str.contains('government of india', na=False)
    mask_remove = mask_blacklist | mask_university_system | mask_govt_india
    
    # Step 4: Remove unwanted records
    df_filtered = df[~mask_remove].copy()
    df_filtered.reset_index(drop=True, inplace=True)
    
    # Step 5: Merge with mapping file
    mapping_df = pd.read_csv(mapping_csv_path)
    merged_data = pd.merge(
        df_filtered,
        mapping_df,
        on='Primary Org Id',
        how='left'
    )
    
    # Step 6: Group ROR IDs by paper ID
    ground_truth = merged_data.groupby('ArXiv Id')['ROR ID'].apply(lambda x: x.dropna().tolist()).to_dict()
    
    # Step 7: Save ground truth JSON to the specified output path
    ground_truth_path = f'{default_output_path}/groundTruth.json'  # Saving under the provided output path

    ground_truth = {remove_version(key): value for key, value in ground_truth.items()}

    with open(ground_truth_path, 'w') as f:
        json.dump(ground_truth, f, indent=4)
    
    print(f"Ground truth JSON saved as '{ground_truth_path}'.")
    
    return ground_truth


In [11]:
import os
scopus_csv_path = os.path.join("../data", "2311_scopus_17416.csv")
default_output_dir = os.path.join("../data", "2311_scopus_17416")
mapping_csv_path = os.path.join("../matching_data", "matched_results_ror_api.csv")
blacklist_path = os.path.join("../data", "blacklist_parent_organizations.txt")

ground_truth = extract_ground_truth(mapping_csv_path, blacklist_path, scopus_csv_path, default_output_dir)

../data/2311_scopus_17416.csv
Ground truth JSON saved as '../data/2311_scopus_17416/groundTruth.json'.
